In [ ]:
import os 
files = os.listdir(
    "paciente 1"
)

In [ ]:
import ollama
import numpy as np
from datetime import datetime
import pymupdf
import json
from codecarbon import OfflineEmissionsTracker
from icecream import ic

In [ ]:
files[1]

In [ ]:
doc = pymupdf.open(f"paciente 1/{files[0]}") # open a document
out = open(f"output_{files[0]}.txt", "wb") # create a text output
tables_all = []
for page in doc: # iterate the document pages
    text = page.get_text().encode("latin-1") # get plain text (is in UTF-8)
    out.write(text) # write text of page
    out.write(bytes((12,))) # write page delimiter (form feed 0x0C)
out.close()

In [ ]:
def AskOllama(prompt, model_name = "gpt-oss:latest"):
    from ollama import chat
    from ollama import ChatResponse

    response: ChatResponse = chat(model=model_name, messages=[
    {
        'role': 'user',
        'content': prompt,
    },
    ])

    return response['message']['content']

In [ ]:
def TranslateOllama(text, code_input, code_output, model = "translategemma:4b"):

    language = {
        "es": "Spanish",
        "en": "English", 

    }

    SOURCE_LANG = language[code_input]
    TARGET_LANG = language[code_output]
    prompt = f"""You are a professional {SOURCE_LANG} ({code_input}) to {TARGET_LANG} ({code_output}) translator. Your goal is to accurately convey the meaning and nuances of the original {SOURCE_LANG} text while adhering to {TARGET_LANG} grammar, vocabulary, and cultural sensitivities.
    Produce only the {TARGET_LANG} translation, without any additional explanations or commentary. Please translate the following {SOURCE_LANG} text into {TARGET_LANG}:


    {text}"""

    response = AskOllama(prompt, model_name= model)

    return response

In [ ]:
def IdentifyOllama(text, model = "translategemma:4b"):

    language = {
        "es": "Spanish",
        "en": "English", 

    }
    prompt = f"""You are a language expert. Tell me the language of the following text:

    {text}.
    
    The possible languages are:

    {language}

    RESPOND ONLY WITH THE LANGUAGE CODE. FOR INSTANCE, IF IT IS IN SPANISH RETURN 'ES'. 

    DON'T ADD ANYTHING ELSE.

    """

    response = AskOllama(prompt, model_name= model)

    return response

# TEXT ANALYSIS

In [ ]:
language_input = IdentifyOllama("Whatcha gonna do when the chips are down, now that the chips are down?").strip()

In [ ]:
language_output = "es"

In [ ]:
TranslateOllama("Whatcha gonna do when the chips are down, now that the chips are down?", language_input, language_output)

In [ ]:
with open(f"output_{files[0]}.txt", "r", encoding="latin-1") as f:
    text_ = f.read()

In [ ]:
tracker = OfflineEmissionsTracker(project_name="Test Ollama", measure_power_secs=0.1)
tracker.start()
models_checked = []
for model in ollama.list()["models"]:
    prompt = f"""

    Del siguiente texto: {text_}

    Sácame como diccionario la siguiente información: 

    - Fecha de la consulta: Fecha en formato '%Y-%m-%d %H:%M:%S'
    - Motivo de la consulta: Breve resumen
    - Datos de la consulta: Por ejemplo, si hay una exploración, sacar los datos físicos de la misma.En caso de que no haya ningún dato devolver un diccionario vacío
    - Información extra: Basándote en los análisis, sácame algun valor que esté fuera de rango (si los hay)

    DEVUELVE SÓLO ESA INFORMACIÓN
    
    """


    size = np.round(model["size"]/(1000*1000*1000),2)
    ic(model["model"])
    ic(size)

    if size < 10 and model["model"] in "medgemma:4b":

        tracker.start_task(f"Ask {model['model']}")
        for i in range(3):
            try:
                response = AskOllama(prompt=prompt, model_name = model["model"])

                dict_ = json.loads(response.replace("python", "").replace("json", "").replace("```",""))
                
                result_filename = f"result_paciente1_{files[0]}_{model['model'].replace(':','_')}_{i}.json"
                
                with open(result_filename, "w") as file:
                    json.dump(dict_,file)
                
                models_checked.append({
                    "model_name": model["model"],
                    "file": result_filename
                })
            except Exception as e:
                print(e)

        tracker.stop_task()
    
emissions = tracker.stop()


In [ ]:
models_checked

In [ ]:
with open(f"result_{model['model'].replace(':','_')}_{i}.json", "w") as file:
    json.dump(dict_,file)

In [ ]:
f"paciente_1_{model['model']}_{i}.json"

In [ ]:


print(f"Emissions : {1000 * emissions} g CO₂")
for task_name, task in tracker._tasks.items():
    print(
        f"Emissions : {1000 * task.emissions_data.emissions} g CO₂ for task {task_name}"
    )
    print(
        f"Energy consumed : { task.emissions_data.energy_consumed} kWh for task {task_name}"
    )
    print(
        f"Duration : { task.emissions_data.duration} (s) for task {task_name}"
    )